# Week 7: Bonus Architecture Experiments

**Advanced Challenge:** Experiment with different architectures and regularization strategies  
**Estimated Time:** 60-90 minutes  
**Scaffolding:** 30% provided (exploration-focused)  
**Prerequisite:** Complete Week 7 post-class exercise first

---

## Overview

You've built a working Fashion-MNIST classifier. Now experiment to find the BEST architecture!

**Experiments:**
1. Width vs Depth: Compare different architectures
2. Dropout Rate Tuning: Find optimal dropout
3. EarlyStopping Patience: Tune stopping criteria
4. L2 Regularization: Alternative to Dropout
5. Combined Strategies: Best of all techniques

**Goal:** Achieve > 90% test accuracy

---

## Setup

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers, regularizers
from keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd

print(f"Keras version: {keras.__version__}")
print("Setup complete!")

In [ ]:
# Load and prepare data (provided)
from keras.datasets import fashion_mnist

(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# Normalize
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

# Three-way split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

# Flatten
X_train_flat = X_train.reshape(-1, 784)
X_val_flat = X_val.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

print(f"Training: {X_train_flat.shape}")
print(f"Validation: {X_val_flat.shape}")
print(f"Test: {X_test_flat.shape}")
print("Data ready!")

---

## Experiment 1: Width vs Depth

Compare:
- **Shallow & Wide:** Fewer layers, more neurons per layer
- **Deep & Narrow:** More layers, fewer neurons per layer

In [ ]:
# TODO: Build shallow & wide model
# Architecture: 512 → 256 → 10
# Add Dropout(0.3) after each Dense layer

model_wide = keras.Sequential([
    layers.Input(shape=(784,)),
    # TODO: Complete architecture
])

model_wide.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Wide model:")
model_wide.summary()

In [ ]:
# Train wide model
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

history_wide = model_wide.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop],
    verbose=0
)

test_acc_wide = model_wide.evaluate(X_test_flat, y_test, verbose=0)[1]
print(f"\nWide Model Test Accuracy: {test_acc_wide:.4f}")

In [ ]:
# TODO: Build deep & narrow model
# Architecture: 256 → 128 → 64 → 32 → 10
# Add Dropout(0.3) after each Dense layer

model_deep = keras.Sequential([
    layers.Input(shape=(784,)),
    # TODO: Complete architecture
])

model_deep.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Deep model:")
model_deep.summary()

In [ ]:
# Train deep model
history_deep = model_deep.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop],
    verbose=0
)

test_acc_deep = model_deep.evaluate(X_test_flat, y_test, verbose=0)[1]
print(f"\nDeep Model Test Accuracy: {test_acc_deep:.4f}")

In [ ]:
# Compare wide vs deep
print("\n" + "="*50)
print("Width vs Depth Comparison")
print("="*50)
print(f"Wide (512→256): {test_acc_wide:.4f}")
print(f"Deep (256→128→64→32): {test_acc_deep:.4f}")
print(f"Winner: {'Wide' if test_acc_wide > test_acc_deep else 'Deep'}")

**Reflection:**
- Which performed better?
- Did deeper model train slower?
- Which has more parameters?

---

## Experiment 2: Dropout Rate Tuning

Test rates: 0.2, 0.3, 0.4, 0.5

In [ ]:
# TODO: Function to build model with configurable dropout rate

def build_model(dropout_rate):
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
# TODO: Test different dropout rates
dropout_rates = [0.2, 0.3, 0.4, 0.5]
results = []

for rate in dropout_rates:
    print(f"\nTesting dropout rate: {rate}")
    
    # Build and train model
    model = build_model(rate)
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    
    history = model.fit(
        X_train_flat, y_train,
        validation_data=(X_val_flat, y_val),
        epochs=50,
        batch_size=128,
        callbacks=[early_stop],
        verbose=0
    )
    
    # Evaluate
    test_acc = model.evaluate(X_test_flat, y_test, verbose=0)[1]
    val_acc = max(history.history['val_accuracy'])
    
    results.append({
        'Dropout Rate': rate,
        'Val Accuracy': val_acc,
        'Test Accuracy': test_acc
    })
    
    print(f"Test Accuracy: {test_acc:.4f}")

# Display results
results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("Dropout Rate Tuning Results")
print("="*50)
print(results_df.to_string(index=False))
print(f"\nBest dropout rate: {results_df.loc[results_df['Test Accuracy'].idxmax(), 'Dropout Rate']}")

**Reflection:**
- Which dropout rate worked best?
- Did higher dropout always improve results?
- What's the tradeoff?

---

## Experiment 3: EarlyStopping Patience Tuning

Test patience values: 3, 5, 7, 10

In [ ]:
# TODO: Test different patience values
patience_values = [3, 5, 7, 10]
results_patience = []

for patience in patience_values:
    print(f"\nTesting patience: {patience}")
    
    # Build model
    model = build_model(dropout_rate=0.3)
    early_stop = EarlyStopping(
        monitor='val_loss', 
        patience=patience, 
        restore_best_weights=True, 
        verbose=0
    )
    
    # Train
    history = model.fit(
        X_train_flat, y_train,
        validation_data=(X_val_flat, y_val),
        epochs=50,
        batch_size=128,
        callbacks=[early_stop],
        verbose=0
    )
    
    # Evaluate
    test_acc = model.evaluate(X_test_flat, y_test, verbose=0)[1]
    epochs_trained = len(history.history['loss'])
    
    results_patience.append({
        'Patience': patience,
        'Epochs Trained': epochs_trained,
        'Test Accuracy': test_acc
    })
    
    print(f"Epochs: {epochs_trained}, Test Accuracy: {test_acc:.4f}")

# Display results
results_patience_df = pd.DataFrame(results_patience)
print("\n" + "="*50)
print("Patience Tuning Results")
print("="*50)
print(results_patience_df.to_string(index=False))

**Reflection:**
- Does more patience = better accuracy?
- What's the cost of higher patience?
- Diminishing returns?

---

## Experiment 4: L2 Regularization

Alternative to Dropout: Add penalty for large weights

In [ ]:
# TODO: Build model with L2 regularization instead of Dropout
# HINT: Use kernel_regularizer=regularizers.l2(0.001) in Dense layers

model_l2 = keras.Sequential([
    layers.Input(shape=(784,)),
    
    # TODO: Add L2 regularization to Dense layers
    # Use regularizers.l2(0.001)
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(____)),
    
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(____)),
    
    layers.Dense(10, activation='softmax')
])

model_l2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model with L2 regularization:")
model_l2.summary()

In [ ]:
# Train L2 model
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

history_l2 = model_l2.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

test_acc_l2 = model_l2.evaluate(X_test_flat, y_test, verbose=0)[1]
print(f"\nL2 Regularization Test Accuracy: {test_acc_l2:.4f}")

**Reflection:**
- How does L2 compare to Dropout?
- Different training behavior?
- Can you combine both?

---

## Experiment 5: Combined Best Practices

**Goal:** Combine everything learned to achieve > 90% accuracy

In [ ]:
# TODO: Build your best model
# Use insights from previous experiments:
# - Best architecture (wide vs deep)
# - Best dropout rate
# - Best patience value
# - Consider adding L2 regularization
# - Try batch normalization: layers.BatchNormalization()

model_best = keras.Sequential([
    layers.Input(shape=(784,)),
    
    # TODO: Design your best architecture here
    # Suggestions:
    # - 3-4 hidden layers
    # - Dropout after each Dense
    # - BatchNormalization for stability
    # - L2 regularization on Dense layers
])

model_best.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Your best model:")
model_best.summary()

In [ ]:
# Train your best model
early_stop_best = EarlyStopping(
    monitor='val_loss',
    patience=7,  # Adjust based on your experiments
    restore_best_weights=True,
    verbose=1
)

history_best = model_best.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),
    epochs=100,  # More epochs with good patience
    batch_size=128,
    callbacks=[early_stop_best],
    verbose=1
)

test_acc_best = model_best.evaluate(X_test_flat, y_test, verbose=0)[1]
print(f"\n{'='*50}")
print(f"BEST MODEL Test Accuracy: {test_acc_best:.4f}")
print(f"{'='*50}")

if test_acc_best >= 0.90:
    print("\nCONGRATULATIONS! You achieved > 90% accuracy!")
else:
    print(f"\nClose! You achieved {test_acc_best:.2%}. Keep experimenting!")

In [ ]:
# Plot final training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_best.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history_best.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Best Model - Loss')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_best.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history_best.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Best Model - Accuracy')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save your best model
model_best.save('week7_best_model.keras')
print("Best model saved!")

---

## Final Summary

**Document your findings:**

In [ ]:
# Compare all experiments
summary = pd.DataFrame([
    {'Experiment': 'Wide Architecture', 'Test Accuracy': test_acc_wide},
    {'Experiment': 'Deep Architecture', 'Test Accuracy': test_acc_deep},
    {'Experiment': 'L2 Regularization', 'Test Accuracy': test_acc_l2},
    {'Experiment': 'BEST MODEL', 'Test Accuracy': test_acc_best}
])

print("\n" + "="*50)
print("All Experiments Summary")
print("="*50)
print(summary.to_string(index=False))
print(f"\nBest Test Accuracy: {summary['Test Accuracy'].max():.4f}")

---

## Key Insights

**What did you learn?**

1. **Width vs Depth:**
   - ___________________________________________________________________

2. **Optimal Dropout Rate:**
   - ___________________________________________________________________

3. **Patience Value:**
   - ___________________________________________________________________

4. **L2 vs Dropout:**
   - ___________________________________________________________________

5. **Best Combination:**
   - ___________________________________________________________________

6. **Biggest Surprise:**
   - ___________________________________________________________________

---

## Additional Challenges

If you want to push further:

1. **Data Augmentation:** Rotate/shift images slightly during training
2. **Learning Rate Scheduling:** Reduce learning rate over time
3. **Ensemble Methods:** Train multiple models and average predictions
4. **Different Optimizers:** Try SGD with momentum, RMSprop
5. **Hyperparameter Search:** Use grid search or random search

---

*Week 7 Bonus Experiments | Version 1.0 | February 2026*